## HW 2 — Co-PC-CO Intermediate: CO Adsorption and Desorption Energy

**Objective**: Model CO binding to Co-PC and compute the CO desorption energy.

### Background

Co-PC can catalyze CO₂ reduction to CO. Once CO is produced, it must desorb from the Co center for the catalyst to turn over. The Co-PC-CO intermediate (CO bound axially to Co) can be a resting state.  
ΔE_des = E(Co-PC) + E(CO) − E(Co-PC-CO)  
gives insight into catalyst poisoning and turnover.

### Tasks

1. **Build the Co-PC-CO Geometry**  
   - Starting from the optimized Co-PC geometry (HW 1), place a CO molecule ~2.0 Å above the Co center along the axial direction (z-axis).  
   - Use ASE or manual coordinate editing to construct the initial guess.

2. **Geometry Optimization of Co-PC-CO**  
   - Optimize Co-PC-CO at B3LYP/def2-SVP.  
   - Report the optimized Co–C(O) bond length and the C–O bond length inside the adduct (compare to free CO: 1.128 Å).  
   - Examine whether spin state changes upon CO binding (doublet Co-PC → singlet or doublet Co-PC-CO?).

3. **Desorption Energy**  
   - Compute ΔE_des = E(Co-PC) + E(CO) − E(Co-PC-CO) at B3LYP/def2-SVP.  
   - Repeat at B3LYP/def2-TZVP for a higher-quality estimate.  
   - Add zero-point energy (ZPE) correction by computing vibrational frequencies for each species.

4. **Thermochemistry at 298 K**  
   - Use the harmonic approximation to compute ΔH and ΔG for desorption at 298 K and 1 atm.  
   - Discuss whether CO is likely to poison the Co-PC catalyst under ambient conditions.

5. *(Optional)* **Potential Energy Surface Along Co–CO Distance**  
   - Scan the Co–C distance from 1.8 Å to 4.0 Å (step 0.1 Å) with all other coordinates relaxed.  
   - Plot E vs. Co–C distance to visualize the binding well and dissociation limit.


In [2]:
import numpy as np
def read_xyz(path):
    with open(path) as f:
        lines = f.readlines()
    atoms = []
    for line in lines[2:]:
        parts = line.split()
        if len(parts) >= 4:
            atoms.append((parts[0], float(parts[1]), float(parts[2]), float(parts[3])))
    return atoms
atoms_opt = read_xyz('CoPC_opt.xyz')
co = [a for a in atoms_opt if a[0] == 'Co'][0]
print(f"Co position: x={co[1]:.3f}, y={co[2]:.3f}, z={co[3]:.3f}")

n_coords = np.array([(a[1], a[2], a[3]) for a in atoms_opt if a[0] == 'N'])
print(f"Mean N position: x={n_coords[:,0].mean():.3f}, y={n_coords[:,1].mean():.3f}, z={n_coords[:,2].mean():.3f}")
print(f"Spread in z: {n_coords[:,2].std():.4f}")

Co position: x=20.035, y=19.815, z=20.988
Mean N position: x=20.034, y=19.815, z=20.988
Spread in z: 1.9446


molecule inst in the xy plane


In [3]:
print(f"Spread in x: {n_coords[:,0].std():.4f}")
print(f"Spread in y: {n_coords[:,1].std():.4f}")
print(f"Spread in z: {n_coords[:,2].std():.4f}")

Spread in x: 1.1012
Spread in y: 1.5929
Spread in z: 1.9446


I need the normal to the copc plane


In [4]:
#plane normal using the 4 coordinating Natoms
coord_n = []
co_pos = np.array([co[1], co[2], co[3]])
for a in atoms_opt:
    if a[0] == 'N':
        d = np.linalg.norm(np.array(a[1:]) - co_pos)
        if d < 2.5:
            coord_n.append(a[1:])

coord_n = np.array(coord_n)
print(f"4 coordinating N positions:\n{coord_n}")

#fit a plane
centered = coord_n - coord_n.mean(axis=0)
_, _, Vt = np.linalg.svd(centered)
normal = Vt[-1]
normal = normal / np.linalg.norm(normal)
print(f"\nPlane normal vector: {normal}")

#place CO 2.0 Å above Co
co_c = co_pos + normal * 2.0 
co_o = co_pos + normal * 3.13
print(f"\nCO placement:")
print(f"  C: {co_c[0]:.6f}  {co_c[1]:.6f}  {co_c[2]:.6f}")
print(f"  O: {co_o[0]:.6f}  {co_o[1]:.6f}  {co_o[2]:.6f}")

#xyz file
with open('CoPCCO.xyz', 'w') as f:
    f.write(f"{len(atoms_opt) + 2}\n")
    f.write("Co-PC-CO initial geometry: CO placed 2.0 A above Co\n")
    for sym, x, y, z in atoms_opt:
        f.write(f"  {sym:2s}  {x:14.8f}  {y:14.8f}  {z:14.8f}\n")
    f.write(f"  C   {co_c[0]:14.8f}  {co_c[1]:14.8f}  {co_c[2]:14.8f}\n")
    f.write(f"  O   {co_o[0]:14.8f}  {co_o[1]:14.8f}  {co_o[2]:14.8f}\n")
print("\nWrote CoPCCO.xyz (59 atoms)")

4 coordinating N positions:
[[20.03778436 19.83338013 22.90960396]
 [18.93551444 18.22582366 21.00450809]
 [20.03122317 19.79696297 19.06560688]
 [21.13337249 21.40457794 20.97070823]]

Plane normal vector: [-0.82254659  0.56868378 -0.0039836 ]

CO placement:
  C: 18.389631  20.952409  20.979668
  O: 17.460153  21.595022  20.975167

Wrote CoPCCO.xyz (59 atoms)


In [5]:
import py3Dmol
def view_structure(filename, width=600, height=450):
    with open(filename, 'r') as f:
        mol_data = f.read()
    view = py3Dmol.view(width=width, height=height)
    view.addModel(mol_data, 'xyz')
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}})
    view.setBackgroundColor('white')
    view.zoomTo()
    return view.show()

view_structure('CoPCCO.xyz')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
#ORCA input
co_input = """! B3LYP def2-SVP def2/J RIJCOSX TightSCF Opt Freq

%maxcore 4000

* xyz 0 1
  C   0.000000   0.000000   0.000000
  O   0.000000   0.000000   1.128000
*
"""

with open('CO_free.inp', 'w') as f:
    f.write(co_input)

print("Wrote CO_free.inp")

Wrote CO_free.inp


In [7]:
#TZVP CO input
co_tzvp = """! B3LYP def2-TZVP def2/J RIJCOSX TightSCF Opt Freq

%maxcore 4000

* xyz 0 1
  C   0.000000   0.000000   0.000000
  O   0.000000   0.000000   1.128000
*
"""

with open('CO_free_tzvp.inp', 'w') as f:
    f.write(co_tzvp)

print("Wrote CO_free_tzvp.inp")

Wrote CO_free_tzvp.inp


In [8]:
#HW2 Task3
#E_des= E(Co-PC) +E(CO) -E(Co-PC-CO)

E_CoPC     = -3047.578120  
E_CO       = -113.172828   
E_CoPCCO   = -3160.799523

HA2KCAL = 627.509

dE_des = (E_CoPC + E_CO - E_CoPCCO) * HA2KCAL

print(f"E(Co-PC):    {E_CoPC:.6f} Eh")
print(f"E(CO):       {E_CO:.6f} Eh")
print(f"E(Co-PC-CO): {E_CoPCCO:.6f} Eh")
print(f"\nΔE_des = {dE_des:.2f} kcal/mol")
print(f"{'Endothermic (CO bound)' if dE_des > 0 else 'Exothermic (CO unbound)'}")

E(Co-PC):    -3047.578120 Eh
E(CO):       -113.172828 Eh
E(Co-PC-CO): -3160.799523 Eh

ΔE_des = 30.48 kcal/mol
Endothermic (CO bound)


In [9]:
atoms_copcco = read_xyz('copcco_opt.xyz')

co = None
last_c = None
last_o = None
for i, a in enumerate(atoms_copcco):
    if a[0] == 'Co':
        co = (i, np.array(a[1:]))
    if a[0] == 'C':
        last_c = (i, np.array(a[1:]))
    if a[0] == 'O':
        last_o = (i, np.array(a[1:]))

co_c_dist = np.linalg.norm(co[1] - last_c[1])
c_o_dist = np.linalg.norm(last_c[1] - last_o[1])

print(f"Co-C(O) bond length: {co_c_dist:.4f} Å")
print(f"C-O bond length:     {c_o_dist:.4f} Å")
print(f"Free CO bond length: 1.128 Å (experimental)")
print(f"C-O elongation upon binding: {(c_o_dist - 1.128)*1000:.1f} mÅ")

Co-C(O) bond length: 2.0388 Å
C-O bond length:     1.1327 Å
Free CO bond length: 1.128 Å (experimental)
C-O elongation upon binding: 4.7 mÅ


In [1]:
#Desorption Energy and Thermochemistry

HA2KCAL = 627.509

E_CoPC_svp    = -3047.578120
E_CO_svp      = -113.172828
E_CoPCCO_svp  = -3160.799525

dE_svp = (E_CoPC_svp + E_CO_svp - E_CoPCCO_svp) * HA2KCAL

E_CoPC_tzvp   = -3049.544709   
E_CO_tzvp     = -113.310353
E_CoPCCO_tzvp = -3162.895573

dE_tzvp = (E_CoPC_tzvp + E_CO_tzvp - E_CoPCCO_tzvp) * HA2KCAL

ZPE_CoPC   = 0.41502557
ZPE_CO     = 0.00510952
ZPE_CoPCCO = 0.42373326

dZPE = (ZPE_CoPC + ZPE_CO - ZPE_CoPCCO) * HA2KCAL

H_CoPC   = -3047.13472996
H_CO     = -113.172828 + 0.00510952 + 0.00236 
H_CoPCCO = -3160.34447780

G_CoPC   = -3047.21689971
G_CoPCCO = -3160.43254427

with open('CO_free.out') as f:
    co_text = f.read()
import re
G_CO_match = re.search(r'Final Gibbs free energy\s+\.\.\.\s+([\-0-9.]+)', co_text)
H_CO_match = re.search(r'Total Enthalpy\s+\.\.\.\s+([\-0-9.]+)', co_text)

if G_CO_match and H_CO_match:
    G_CO = float(G_CO_match.group(1))
    H_CO = float(H_CO_match.group(1))
else:
    print("WARNING: Could not find CO thermochemistry. Check CO_free.out has Freq data.")
    G_CO = None
    H_CO = None

print("=" * 60)
print("HW2: CO Desorption from Co-PC")
print("=" * 60)

print(f"\n--- Electronic Desorption Energy ---")
print(f"  def2-SVP:   dE = {dE_svp:.2f} kcal/mol")
print(f"  def2-TZVP:  dE = {dE_tzvp:.2f} kcal/mol")

print(f"\n--- ZPE Correction (SVP) ---")
print(f"  dZPE = {dZPE:.2f} kcal/mol")
print(f"  dE(SVP) + dZPE = {dE_svp + dZPE:.2f} kcal/mol")

if H_CO and G_CO:
    dH = (H_CoPC + H_CO - H_CoPCCO) * HA2KCAL
    dG = (G_CoPC + G_CO - G_CoPCCO) * HA2KCAL
    print(f"\n--- Thermochemistry at 298 K, 1 atm (SVP) ---")
    print(f"  dH = {dH:.2f} kcal/mol")
    print(f"  dG = {dG:.2f} kcal/mol")
    print(f"\n--- Interpretation ---")
    if dG > 0:
        print(f"  dG > 0: CO desorption is thermodynamically unfavorable at 298 K.")
        print(f"  CO remains bound -> potential catalyst poisoning.")
    else:
        print(f"  dG < 0: CO desorption is thermodynamically favorable at 298 K.")
        print(f"  Catalyst can turn over under ambient conditions.")

HW2: CO Desorption from Co-PC

--- Electronic Desorption Energy ---
  def2-SVP:   dE = 30.48 kcal/mol
  def2-TZVP:  dE = 25.42 kcal/mol

--- ZPE Correction (SVP) ---
  dZPE = -2.26 kcal/mol
  dE(SVP) + dZPE = 28.22 kcal/mol

--- Thermochemistry at 298 K, 1 atm (SVP) ---
  dH = 28.45 kcal/mol
  dG = 18.07 kcal/mol

--- Interpretation ---
  dG > 0: CO desorption is thermodynamically unfavorable at 298 K.
  CO remains bound -> potential catalyst poisoning.


# HW 2 — Co-PC-CO Intermediate: CO Adsorption and Desorption Energy

## Task 1: Building the Co-PC-CO Geometry

The optimized CoPc geometry from HW 1 served as the starting point. Because the molecule was not aligned to any Cartesian axis (the original coordinates were offset around ~20 Angstrom in all three dimensions), the molecular plane normal had to be computed explicitly. The four coordinating nitrogen positions were centered and decomposed via SVD; the smallest singular vector gave the plane normal. A CO molecule was then placed along this normal, with the carbon atom 2.0 Angstrom above cobalt and the oxygen a further 1.13 Angstrom beyond that (matching the free CO bond length as an initial guess). The resulting 59-atom structure was written to CoPCCO.xyz and confirmed visually in py3Dmol — the CO unit protrudes axially from the flat CoPc ring, as expected for a five-coordinate Co adduct.

## Task 2: Geometry Optimization of Co-PC-CO

The Co-PC-CO structure was optimized at B3LYP/def2-SVP (RIJCOSX, doublet) using ORCA 6.1.0 on HPC. The calculation converged normally, yielding E = -3160.7995 Eh.

The optimized bond lengths are:

| Bond | Length |
|------|--------|
| Co-C(O) | 2.039 Angstrom |
| C-O (in adduct) | 1.133 Angstrom |
| C-O (free CO) | 1.128 Angstrom |

The C-O bond elongates by 4.7 milliangstrom upon binding to cobalt. This is modest but physically meaningful: it reflects back-donation from occupied Co d-orbitals (primarily dxz and dyz) into the CO pi* antibonding orbital, which weakens the C-O bond. The small magnitude of the elongation is consistent with CO being a strong-field ligand that accepts back-donation without dramatically altering its internal bond length.

The calculation was run as a doublet (multiplicity 2), maintaining the same spin state as bare CoPc. CO is a closed-shell, strong-field ligand, so it is not expected to alter the spin state — and indeed the SCF converged smoothly without spin contamination issues, suggesting the doublet remains the ground state for the adduct.

## Task 3: Desorption Energy

The CO desorption energy was computed as:

$$\Delta E_{\text{des}} = E(\text{Co-PC}) + E(\text{CO}) - E(\text{Co-PC-CO})$$

at both def2-SVP and def2-TZVP levels:

| Basis Set | E(Co-PC) (Eh) | E(CO) (Eh) | E(Co-PC-CO) (Eh) | Delta E (kcal/mol) |
|-----------|--------------|-----------|-----------------|-------------------|
| def2-SVP | -3047.5781 | -113.1728 | -3160.7995 | 30.48 |
| def2-TZVP | -3049.5447 | -113.3104 | -3162.8956 | 25.42 |

The SVP value of 30.5 kcal/mol drops to 25.4 kcal/mol at TZVP, a reduction of about 5 kcal/mol. This is the expected basis set superposition effect: the smaller SVP basis artificially stabilizes the complex relative to the separated fragments, inflating the binding energy. The TZVP result is more reliable.

Zero-point energy corrections were obtained from frequency calculations at def2-SVP for all three species (Co-PC, CO, and Co-PC-CO). The ZPE correction to the desorption energy is:

$$\Delta\text{ZPE} = \text{ZPE}(\text{Co-PC}) + \text{ZPE}(\text{CO}) - \text{ZPE}(\text{Co-PC-CO}) = -2.26 \text{ kcal/mol}$$

The negative sign means the complex has slightly more zero-point energy than the separated fragments (the new Co-C stretching and bending modes add vibrational energy). Applying this correction gives a ZPE-corrected desorption energy of 28.22 kcal/mol at def2-SVP.

## Task 4: Thermochemistry at 298 K

Using the harmonic-oscillator / rigid-rotor / ideal-gas approximation as implemented in ORCA's thermochemistry module (298.15 K, 1 atm), the enthalpy and Gibbs free energy of desorption were computed from the frequency calculations:

$$\Delta H_{\text{des}} = H(\text{Co-PC}) + H(\text{CO}) - H(\text{Co-PC-CO}) = +28.45 \text{ kcal/mol}$$

$$\Delta G_{\text{des}} = G(\text{Co-PC}) + G(\text{CO}) - G(\text{Co-PC-CO}) = +18.07 \text{ kcal/mol}$$

The entropy contribution (T times Delta S) accounts for the ~10 kcal/mol difference between Delta H and Delta G. This is almost entirely translational and rotational entropy gained by the free CO molecule upon desorption — a small, rigid diatomic gains substantial entropy when released from a large, heavy complex. Despite this favorable entropic contribution, Delta G remains strongly positive at +18 kcal/mol.

A positive Delta G of this magnitude means CO desorption is thermodynamically unfavorable at room temperature. Under ambient conditions, the Co-PC-CO adduct is the stable state, and the catalyst will not spontaneously release CO. In the context of CO2 reduction catalysis, this implies that CoPc is susceptible to product inhibition: the CO produced during the catalytic cycle binds strongly enough to the active site that turnover is impeded. Overcoming this would require either elevated temperature, a driving electrochemical potential, or catalyst modification to weaken the Co-CO interaction — for instance, introducing electron-withdrawing substituents on the phthalocyanine ring to reduce back-donation into CO pi*.

Used an LLM to use my interpretted results to format the write up
